In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. SETUP PARAMETERS
DATA_DIR = "Data"
COUNTRIES = ["Belgium", "Spain", "Poland"]
# These are the task codes for Non-Routine Cognitive Analytical tasks
TASKS = ["t_4A2a4", "t_4A2b2", "t_4A4a1"]

# 2. HELPER FUNCTION
def get_weighted_std(df, target_col, weight_col):
    """Calculates weighted standardization."""
    w_mean = np.average(df[target_col], weights=df[weight_col])
    w_var = np.average((df[target_col] - w_mean)**2, weights=df[weight_col])
    return (df[target_col] - w_mean) / np.sqrt(w_var)

# 3. DATA LOADING
# Make sure the 'Data' folder exists with these files inside!
task_path = os.path.join(DATA_DIR, "onet_tasks.csv")
isco_path = os.path.join(DATA_DIR, "Eurostat_employment_isco.xlsx")

task_data = pd.read_csv(task_path)

# Loop to load all ISCO sheets (ISCO1 to ISCO9)
isco_list = []
for i in range(1, 10):
    # Added engine='openpyxl' to be safe with newer Excel files
    temp_df = pd.read_excel(isco_path, sheet_name=f"ISCO{i}", engine='openpyxl')
    temp_df['ISCO'] = i
    isco_list.append(temp_df)

all_data = pd.concat(isco_list, ignore_index=True)

# 4. PROCESSING & MERGING
for c in COUNTRIES:
    # Calculate shares: (workers in ISCO category) / (total workers in country at that time)
    all_data[f'total_{c}'] = all_data.groupby('TIME')[c].transform('sum')
    all_data[f'share_{c}'] = all_data[c] / all_data[f'total_{c}']

# Extract 1-digit ISCO from task data (e.g., 1110 -> 1)
task_data["isco08_1dig"] = task_data["isco08"].astype(str).str[:1].astype(int)
aggdata = task_data.groupby("isco08_1dig")[TASKS].mean().reset_index()

combined = pd.merge(all_data, aggdata, left_on='ISCO', right_on='isco08_1dig', how='left')

# 5. NRCA CALCULATIONS
for c in COUNTRIES:
    share_col = f'share_{c}'
    
    # Standardize each task using the function
    for t in TASKS:
        combined[f'std_{c}_{t}'] = get_weighted_std(combined, t, share_col)
    
    # Sum standardized tasks to get composite NRCA
    std_cols = [f'std_{c}_{t}' for t in TASKS]
    combined[f'{c}_NRCA'] = combined[std_cols].sum(axis=1)
    
    # Final standardized NRCA and weighted multiplier
    combined[f'std_{c}_NRCA'] = get_weighted_std(combined, f'{c}_NRCA', share_col)
    combined[f'multip_{c}_NRCA'] = combined[f'std_{c}_NRCA'] * combined[share_col]

# 6. PLOTTING
for c in COUNTRIES:
    plot_data = combined.groupby("TIME")[f"multip_{c}_NRCA"].sum().reset_index()
    
    plt.figure(figsize=(10, 4))
    plt.plot(plot_data["TIME"], plot_data[f"multip_{c}_NRCA"], marker='o', color='blue')
    plt.title(f"NRCA Intensity over Time: {c}")
    plt.xlabel("Quarter")
    plt.ylabel("NRCA Index")
    # Show every 4th label to avoid crowding the X-axis
    plt.xticks(plot_data["TIME"][::4], rotation=45) 
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()